# Chapter 13a — Shared Memory & the Dot Product (companion)

> Companion to **Chapter 13 — Block Reductions**, distilled from *CUDA by Example*
> (Sanders & Kandrot), **Chapter 5 — Thread Cooperation**.
> A gentler on-ramp to the block-level reductions you met in Chapter 13.

Chapters 12–13 jumped straight to per-row reductions for LayerNorm and softmax. This companion slows down and builds the one primitive that makes block-level reductions possible: [**shared memory**](https://developer.nvidia.com/blog/using-shared-memory-cuda-cc/) — a small, fast, on-chip scratchpad that every thread in a block can read and write, so threads can *cooperate* instead of each working alone.

The book's vehicle for this is the **dot product**: `sum(a[i] * b[i])`. It's the simplest computation that *can't* be done one-output-per-thread — every thread produces a partial result that must be combined with its neighbours'. That combination is a **reduction**, and shared memory is where it happens.

### Learning objectives

By the end you will:

- Explain what `__shared__` memory is (scope, lifetime, speed) vs. global memory.
- Use `__syncthreads()` to coordinate threads within a block — and explain the **divergence trap** that makes a wrong `__syncthreads()` hang the kernel.
- Write a **parallel tree reduction** in shared memory (`log2(N)` steps, not `N`).
- Connect the book's `cache[]` reduction to `llm.c`'s block reductions in `layernorm.cuh` and `global_norm.cu` (Chapter 13).


## 1. Concept — Shared Memory Is a Per-Block Scratchpad

So far every thread read from and wrote to **global memory** (the GPU's DRAM, `cudaMalloc`'d). Global memory is large (16 GB on the 4080 SUPER) but relatively far away (~hundreds of cycles).

**Shared memory** is different:

| | Global memory | Shared memory |
|---|---|---|
| Declared | `cudaMalloc` | `__shared__` inside a kernel |
| Scope | all threads, all blocks | **one block only** |
| Lifetime | whole program | **lifetime of the block** |
| Speed | ~hundreds of cycles | ~a few cycles (on-chip, SRAM) |
| Size | GBs | ~48–100 KB **per block** |

The defining property: **threads in the same block share it.** Thread 5 can write `cache[5]` and thread 6 can read it. That's the whole point — it's the channel through which threads *cooperate*. Threads in *different* blocks cannot see each other's shared memory; for that you need global memory or atomics (Chapter 13b).

```c
__global__ void k(...) {
    __shared__ float cache[256];   // one copy per block, 256 floats
    cache[threadIdx.x] = ...;      // each thread writes its own slot
    __syncthreads();               // wait until ALL threads have written
    // now any thread can safely read any other thread's slot
}
```

The `__syncthreads()` is mandatory: without it, thread 6 might read `cache[5]` before thread 5 has written it. It's a **barrier** — every thread in the block waits there until all of them arrive.


## 2. The Dot Product — Why One Thread Per Output Doesn't Work

`dot(a, b) = a[0]*b[0] + a[1]*b[1] + ... + a[N-1]*b[N-1]` is a single number summed from `N` products. The element-wise multiply is embarrassingly parallel; the **sum** is the hard part.

The book's strategy (and `llm.c`'s) is two-level:

1. **Each block** computes a *partial* sum over its slice of the data, reducing it to one number using shared memory.
2. The handful of per-block partials are summed — by the CPU in the book's version, or by a second kernel / atomics in production.

Inside a block the reduction is a **tree**: instead of one thread adding 256 numbers serially (256 steps), all threads cooperate to do it in `log2(256) = 8` steps.

```
step 0:  c0 c1 c2 c3 c4 c5 c6 c7      (8 partials in shared memory)
         c0+=c4  c1+=c5  c2+=c6  c3+=c7
step 1:  c0 c1 c2 c3
         c0+=c2  c1+=c3
step 2:  c0 c1
         c0+=c1
result:  c0   ← the block's partial sum
```

Each step halves the number of live values. Let's write it.


In [ ]:
!mkdir -p course/ch13a_build


In [ ]:
%%writefile course/ch13a_build/dot.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

// CUDA by Example, Ch.5 dot product — modernized slightly for clarity.
#define imin(a,b) ((a)<(b)?(a):(b))

const int N              = 33 * 1024;   // not a multiple of the block size, on purpose
const int threadsPerBlock = 256;        // power of 2 (required by the tree reduction)
// cap the grid so the per-block partials fit in a small array the CPU finishes
const int blocksPerGrid  = imin(32, (N + threadsPerBlock - 1) / threadsPerBlock);

__global__ void dot(const float* a, const float* b, float* partials) {
    __shared__ float cache[threadsPerBlock];

    int tid        = blockIdx.x * blockDim.x + threadIdx.x;
    int cacheIndex = threadIdx.x;

    // (1) each thread sums a strided slice of the products into a register
    float temp = 0.0f;
    while (tid < N) {
        temp += a[tid] * b[tid];
        tid  += blockDim.x * gridDim.x;     // grid-stride (Chapter 10)
    }
    cache[cacheIndex] = temp;               // (2) stash into shared memory
    __syncthreads();                        // every thread has written its slot

    // (3) tree reduction in shared memory: log2(threadsPerBlock) steps
    int i = blockDim.x / 2;
    while (i != 0) {
        if (cacheIndex < i)
            cache[cacheIndex] += cache[cacheIndex + i];
        __syncthreads();                    // NOTE: outside the if — see section 3
        i /= 2;
    }

    // (4) thread 0 writes this block's partial to global memory
    if (cacheIndex == 0) partials[blockIdx.x] = cache[0];
}

int main(void) {
    float *a = (float*)malloc(N*sizeof(float));
    float *b = (float*)malloc(N*sizeof(float));
    float *partials = (float*)malloc(blocksPerGrid*sizeof(float));

    for (int i = 0; i < N; i++) { a[i] = i; b[i] = i * 2.0f; }

    float *d_a, *d_b, *d_partials;
    cudaMalloc(&d_a, N*sizeof(float));
    cudaMalloc(&d_b, N*sizeof(float));
    cudaMalloc(&d_partials, blocksPerGrid*sizeof(float));
    cudaMemcpy(d_a, a, N*sizeof(float), cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, N*sizeof(float), cudaMemcpyHostToDevice);

    dot<<<blocksPerGrid, threadsPerBlock>>>(d_a, d_b, d_partials);
    cudaMemcpy(partials, d_partials, blocksPerGrid*sizeof(float), cudaMemcpyDeviceToHost);

    // CPU finishes the final sum of the (<=32) per-block partials
    double gpu = 0.0;
    for (int i = 0; i < blocksPerGrid; i++) gpu += partials[i];

    // reference: sum_{i=0}^{N-1} i*(2i) = 2 * sum i^2
    double cpu = 0.0;
    for (int i = 0; i < N; i++) cpu += (double)a[i] * b[i];

    printf("GPU dot = %.6e\n", gpu);
    printf("CPU dot = %.6e\n", cpu);
    printf("rel diff = %.2e  -> %s\n", fabs(gpu - cpu) / cpu,
           (fabs(gpu - cpu) / cpu < 1e-5) ? "PASS" : "FAIL");

    cudaFree(d_a); cudaFree(d_b); cudaFree(d_partials);
    free(a); free(b); free(partials);
    return 0;
}


In [ ]:
!nvcc -O2 -o course/ch13a_build/dot course/ch13a_build/dot.cu && ./course/ch13a_build/dot


You should see `PASS` with a tiny relative difference (float summation order differs between GPU and CPU, so it won't be bit-exact, but it's within ~1e-6).

The structure to remember — it's the same one `llm.c` uses everywhere:

1. **Grid-stride accumulate** into a per-thread register (`temp`).
2. **Stash** into `cache[threadIdx.x]`, then `__syncthreads()`.
3. **Tree-reduce** the cache in `log2` steps, `__syncthreads()` each step.
4. **One thread** writes the block's result out.


## 3. The `__syncthreads()` Divergence Trap

Look again at the reduction loop:

```c
int i = blockDim.x / 2;
while (i != 0) {
    if (cacheIndex < i)
        cache[cacheIndex] += cache[cacheIndex + i];
    __syncthreads();           // <-- OUTSIDE the if
    i /= 2;
}
```

The `__syncthreads()` is deliberately placed **outside** the `if (cacheIndex < i)`. It is tempting to "optimize" by moving it inside:

```c
if (cacheIndex < i) {
    cache[cacheIndex] += cache[cacheIndex + i];
    __syncthreads();           // <-- WRONG: divergent barrier
}
```

This **hangs the kernel** (or worse, silently corrupts results). `__syncthreads()` requires *every* thread in the block to reach *that* barrier. When it sits inside a divergent branch, the threads where `cacheIndex >= i` never reach it — so the threads that did reach it wait forever. The book calls the consequences "somewhat tragic": the hardware will not let a thread past the barrier until all threads arrive, and some never will.

**Rule: `__syncthreads()` must be reached by all threads in the block — never put it in a branch that only some threads take.** Notice that the threads with `cacheIndex >= i` do a small amount of useless waiting; that's fine. Correctness beats cleverness.


## 4. Translation Bridge — From `cache[]` to `llm.c` Block Reductions

The dot product's shared-memory reduction is *exactly* the pattern behind Chapter 13's LayerNorm and `global_norm`.

| *CUDA by Example* (Ch5) | `llm.c` (Chapter 13) | Role |
|---|---|---|
| `__shared__ float cache[threadsPerBlock]` | `__shared__` scratch in `layernorm_forward_kernel`, `global_norm_squared_kernel` | per-block scratchpad |
| `cache[tid] += cache[tid+i]` tree loop | `blockReduce<...>()` helper in `llmc/cuda_utils.cuh` | combine partials within a block |
| CPU sums the `blocksPerGrid` partials | second kernel / `atomicAdd` (Chapter 13b) | combine partials across blocks |
| `temp += a[tid]*b[tid]` grid-stride | per-row sum of `x` and `x*x` for mean/variance | the actual work |

Two differences in production `llm.c`:

1. It reduces **within a warp first** using `__shfl_down_sync` (Chapter 12) — no shared memory needed for the last 32 lanes — then across warps via shared memory. Fewer `__syncthreads()`, faster.
2. It rarely bounces to the CPU; the cross-block combine is an `atomicAdd` or a second pass, keeping everything on the GPU.

So Chapter 12 (warp shuffles) + this chapter (shared memory) are the two halves of every real block reduction you'll read in the repo.


## 5. Common Pitfalls

- **`threadsPerBlock` must be a power of 2** for the halving tree reduction. 256, 128, 1024 are fine; 200 is not (the `i /= 2` walk would skip elements).
- **Forgetting `__syncthreads()`** between writing `cache[]` and reading a neighbour's slot → race condition, nondeterministic results.
- **`__syncthreads()` in a divergent branch** → deadlock (section 3).
- **Shared memory is per-block and small** (~48 KB default). `__shared__ float cache[100000]` won't launch.
- **Bank conflicts**: shared memory has 32 banks; if many threads hit the same bank with different addresses, accesses serialize. The contiguous `cache[threadIdx.x]` pattern here is conflict-free — but it's why stride matters for shared memory too.


## 6. TODO Exercise — Shared-Memory Max Reduction

Adapt the tree reduction to compute the **maximum** of an array instead of the sum — the exact primitive softmax needs for its per-row max (Chapter 16). Fill in the two TODOs.


In [ ]:
%%writefile course/ch13a_build/exercise1.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

const int N               = 16 * 1024;
const int threadsPerBlock = 256;
const int blocksPerGrid   = 32;

__global__ void block_max(const float* in, float* partials, int n) {
    __shared__ float cache[threadsPerBlock];
    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    int c   = threadIdx.x;

    float m = -1e30f;
    while (tid < n) { m = fmaxf(m, in[tid]); tid += blockDim.x * gridDim.x; }
    cache[c] = m;
    __syncthreads();

    int i = blockDim.x / 2;
    while (i != 0) {
        // TODO 1: if (c < i) combine cache[c] and cache[c+i] with fmaxf
        if (c < i) cache[c] = fmaxf(cache[c], cache[c+i]);
        __syncthreads();
        i /= 2;
    }
    // TODO 2: thread 0 writes cache[0] to partials[blockIdx.x]
    if (c == 0) partials[blockIdx.x] = cache[0]
}

int main(void) {
    float* in = (float*)malloc(N*sizeof(float));
    float* partials = (float*)malloc(blocksPerGrid*sizeof(float));
    float truth = -1e30f;
    for (int i = 0; i < N; i++) { in[i] = sinf(i*0.7f)*100.0f; truth = fmaxf(truth, in[i]); }

    float *d_in, *d_p;
    cudaMalloc(&d_in, N*sizeof(float)); cudaMalloc(&d_p, blocksPerGrid*sizeof(float));
    cudaMemcpy(d_in, in, N*sizeof(float), cudaMemcpyHostToDevice);
    block_max<<<blocksPerGrid, threadsPerBlock>>>(d_in, d_p, N);
    cudaMemcpy(partials, d_p, blocksPerGrid*sizeof(float), cudaMemcpyDeviceToHost);

    float gpu = -1e30f;
    for (int i = 0; i < blocksPerGrid; i++) gpu = fmaxf(gpu, partials[i]);
    printf("GPU max = %.4f   CPU max = %.4f   -> %s\n", gpu, truth,
           (fabs(gpu - truth) < 1e-3) ? "PASS" : "FAIL");
    cudaFree(d_in); cudaFree(d_p); free(in); free(partials);
    return 0;
}


In [ ]:
!nvcc -O2 -o course/ch13a_build/exercise1 course/ch13a_build/exercise1.cu && ./course/ch13a_build/exercise1


### Solution

In [ ]:
%%writefile course/ch13a_build/exercise1_sol.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

const int N               = 16 * 1024;
const int threadsPerBlock = 256;
const int blocksPerGrid   = 32;

__global__ void block_max(const float* in, float* partials, int n) {
    __shared__ float cache[threadsPerBlock];
    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    int c   = threadIdx.x;

    float m = -1e30f;
    while (tid < n) { m = fmaxf(m, in[tid]); tid += blockDim.x * gridDim.x; }
    cache[c] = m;
    __syncthreads();

    int i = blockDim.x / 2;
    while (i != 0) {
        if (c < i) cache[c] = fmaxf(cache[c], cache[c + i]);   // TODO 1
        __syncthreads();
        i /= 2;
    }
    if (c == 0) partials[blockIdx.x] = cache[0];               // TODO 2
}

int main(void) {
    float* in = (float*)malloc(N*sizeof(float));
    float* partials = (float*)malloc(blocksPerGrid*sizeof(float));
    float truth = -1e30f;
    for (int i = 0; i < N; i++) { in[i] = sinf(i*0.7f)*100.0f; truth = fmaxf(truth, in[i]); }

    float *d_in, *d_p;
    cudaMalloc(&d_in, N*sizeof(float)); cudaMalloc(&d_p, blocksPerGrid*sizeof(float));
    cudaMemcpy(d_in, in, N*sizeof(float), cudaMemcpyHostToDevice);
    block_max<<<blocksPerGrid, threadsPerBlock>>>(d_in, d_p, N);
    cudaMemcpy(partials, d_p, blocksPerGrid*sizeof(float), cudaMemcpyDeviceToHost);

    float gpu = -1e30f;
    for (int i = 0; i < blocksPerGrid; i++) gpu = fmaxf(gpu, partials[i]);
    printf("GPU max = %.4f   CPU max = %.4f   -> %s\n", gpu, truth,
           (fabs(gpu - truth) < 1e-3) ? "PASS" : "FAIL");
    cudaFree(d_in); cudaFree(d_p); free(in); free(partials);
    return 0;
}


In [ ]:
!nvcc -O2 -o course/ch13a_build/exercise1_sol course/ch13a_build/exercise1_sol.cu && ./course/ch13a_build/exercise1_sol


## Further Reading

**Adapted from** *CUDA by Example* (Sanders & Kandrot), **Chapter 5 — Thread Cooperation** (the `dot` product).

**Source of truth**

- [_Using Shared Memory in CUDA C/C++_](https://developer.nvidia.com/blog/using-shared-memory-cuda-cc/) (Mark Harris) — the modern reference for `__shared__`, `__syncthreads()`, and bank behavior.
- [_Faster Parallel Reductions on Kepler_](https://developer.nvidia.com/blog/faster-parallel-reductions-kepler/) — how production `llm.c` replaces the book's tree reduction with warp shuffles (Chapter 12–13).

**Going deeper**

- [CUDA C++ Programming Guide — Shared Memory](https://docs.nvidia.com/cuda/cuda-c-programming-guide/index.html) — scope, lifetime, and the `__syncthreads()` contract.


## Recap

- **Shared memory** (`__shared__`) is a small, fast, on-chip scratchpad shared by all threads **in one block** — the channel for thread cooperation.
- **`__syncthreads()`** is a block-wide barrier; it must be reached by **every** thread, so never place it in a divergent branch.
- A **tree reduction** in shared memory combines `threadsPerBlock` values in `log2` steps — the engine behind every block reduction.
- `llm.c` uses exactly this pattern (often warp-shuffle first, shared memory second) in `layernorm.cuh`, `global_norm.cu`, and friends.

### What's next

**Chapter 13b — Atomics & Histograms** (book Ch9). The dot product punted the final cross-block sum to the CPU. Atomics let blocks combine their partials **on the GPU** without races — the mechanism behind `encoder_backward`'s scatter-add and `global_norm`'s accumulation. We'll build a histogram and see why naive atomics are slow and how shared-memory privatization fixes them.
